# TikTok GraphRAG Embedding Pipeline (production)

This notebook:

- Embeds `TikTokVideo.comment_summary_description` → `comment_summary_embedding`
- Embeds combined video text → `video_content_embedding`
- **Syncs comment topics from MongoDB** (`tiktok_user_video`) into Neo4j as `(:TikTokCommentTopic)` and `[:HAS_COMMENT_TOPIC]`, **only** when `(:TikTokVideo {video_id})` already exists with the **same `video_id` string** as Mongo.
- Embeds `TikTokCommentTopic.name` → `embedding` and ensures a dedicated vector index.

Uses Neo4j / Mongo / embedding credentials from `.env`. Re-runs are idempotent.
> **Run order:** After setup and the **Video comment summary embedding functions** cell, scroll to the heading **Run comment summary embeddings (embedding pass)** and run the code cell below it (`# === COMMENT SUMMARY EMBEDDING PASS ===`). Then **video content embedding**, then **topic sync + topic embeddings** (immediately before `driver.close()`).


In [ ]:
import json
import os
import time
from typing import Any, Dict, Iterable, Iterator, List, Optional

from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import AzureOpenAI, OpenAI
from pymongo import MongoClient

load_dotenv()
print('Env loaded')



Env loaded


In [ ]:
def _redact_bolt_uri(uri: str) -> str:
    if not uri:
        return ''
    if '@' not in uri:
        return uri
    left, right = uri.rsplit('@', 1)
    if '://' in left:
        scheme = left.split('://', 1)[0]
        return f'{scheme}://***:***@{right}'
    return uri


def _redact_mongo_uri(uri: str) -> str:
    if not uri or '://' not in uri:
        return uri
    scheme, rest = uri.split('://', 1)
    if '@' not in rest:
        return uri
    _creds, hostpart = rest.rsplit('@', 1)
    return f'{scheme}://***:***@{hostpart}'


# Production Neo4j from .env
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USER = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

# Mongo (same collection as DAG / tiktok_embedding_test)
MONGO_URI = os.getenv('MONGODB_URI') or os.getenv('MONGO_URI')
MONGO_DB = os.getenv('MONGODB_DB') or os.getenv('MONGO_DB', 'rbl')
MONGO_COLLECTION = os.getenv('MONGO_COLLECTION', 'tiktok_user_video')

# Embeddings config from .env
AZURE_OPENAI_EMBEDDING_ENDPOINT = os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT')
AZURE_OPENAI_EMBEDDING_API_KEY = os.getenv('AZURE_OPENAI_EMBEDDING_API_KEY')
AZURE_OPENAI_EMBEDDING_API_VERSION = os.getenv('AZURE_OPENAI_EMBEDDING_API_VERSION', '2024-02-01')
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = (
    os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_MODEL_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
    or 'text-embedding-3-large'
)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-large')

# Tuning knobs
PAGE_SIZE = int(os.getenv('TT_SUMMARY_EMBED_PAGE_SIZE', '500'))
EMBED_BATCH = int(os.getenv('TT_SUMMARY_EMBED_BATCH', '128'))
WRITE_BATCH = int(os.getenv('TT_SUMMARY_WRITE_BATCH', '500'))  # legacy name; vector writes use *_VECTOR_WRITE_BATCH
SUMMARY_VECTOR_WRITE_BATCH = int(os.getenv('TT_SUMMARY_VECTOR_WRITE_BATCH', '32'))  # small txs for Aura (timeout-safe)
CONTENT_VECTOR_WRITE_BATCH = int(os.getenv('TT_CONTENT_VECTOR_WRITE_BATCH', '32'))
NEO4J_CONNECTION_TIMEOUT = float(os.getenv('NEO4J_CONNECTION_TIMEOUT', '120'))  # seconds (Bolt TCP/TLS)
TOPIC_UPSERT_BATCH = int(os.getenv('TT_TOPIC_UPSERT_BATCH', '200'))
TOPIC_SYNC_LIMIT = int(os.getenv('TT_TOPIC_SYNC_LIMIT', '0'))  # 0 = scan full collection
TOPIC_EMBED_WRITE_BATCH = int(os.getenv('TT_TOPIC_EMBED_WRITE_BATCH', '32'))  # small txs for topic vector writes (Aura / prod)

TIKTOK_PLATFORM = 'tiktok'

assert NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD, 'Set NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD in .env'
assert MONGO_URI, 'Set MONGODB_URI or MONGO_URI in .env for topic sync'

print('Config:')
print(f'  neo4j: {_redact_bolt_uri(NEO4J_URI)}  db={NEO4J_DATABASE}  user={NEO4J_USER}')
print(f'  mongo: {_redact_mongo_uri(MONGO_URI)}  db={MONGO_DB}  collection={MONGO_COLLECTION}')
print(f'  batches: page={PAGE_SIZE} embed={EMBED_BATCH} write={WRITE_BATCH} summary_vec={SUMMARY_VECTOR_WRITE_BATCH} content_vec={CONTENT_VECTOR_WRITE_BATCH} topic_upsert={TOPIC_UPSERT_BATCH}  topic_sync_limit={TOPIC_SYNC_LIMIT or "none"}  topic_vec_write={TOPIC_EMBED_WRITE_BATCH}  neo4j_conn_timeout={NEO4J_CONNECTION_TIMEOUT}s')



Config:
  neo4j: bolt+ssc://rbl-neo4j.ecda.ai:7687  db=neo4j  user=neo4j
  mongo: mongodb://***:***@  db=rbl  collection=tiktok_user_video
  batches: page=500 embed=128 write=500 summary_vec=32 content_vec=32 topic_upsert=200  topic_sync_limit=none  topic_vec_write=32  neo4j_conn_timeout=120.0s


In [ ]:
def build_embedding_client():
    if AZURE_OPENAI_EMBEDDING_ENDPOINT and AZURE_OPENAI_EMBEDDING_API_KEY:
        client = AzureOpenAI(
            azure_endpoint=AZURE_OPENAI_EMBEDDING_ENDPOINT,
            api_key=AZURE_OPENAI_EMBEDDING_API_KEY,
            api_version=AZURE_OPENAI_EMBEDDING_API_VERSION,
        )
        return 'azure', client, AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    if OPENAI_API_KEY:
        return 'openai', OpenAI(api_key=OPENAI_API_KEY), OPENAI_EMBEDDING_MODEL
    raise RuntimeError('No embedding credentials found in .env')


def batched(iterable: Iterable, n: int) -> Iterator[list]:
    bucket = []
    for item in iterable:
        bucket.append(item)
        if len(bucket) >= n:
            yield bucket
            bucket = []
    if bucket:
        yield bucket


def _video_id_str(raw_id: Any) -> str:
    if raw_id is None:
        return ''
    if isinstance(raw_id, dict) and '$numberLong' in raw_id:
        return str(raw_id['$numberLong']).strip()
    return str(raw_id).strip()


EMBED_PROVIDER, embed_client, EMBED_MODEL_NAME = build_embedding_client()
probe = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=['ping'])
EMBED_DIM = len(probe.data[0].embedding)

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    connection_timeout=NEO4J_CONNECTION_TIMEOUT,
)
with driver.session(database=NEO4J_DATABASE) as s:
    print('Neo4j ok:', s.run('RETURN 1 AS ok').single()['ok'])
    pending = s.run(
        'MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL AND v.comment_summary_embedding IS NULL RETURN count(v) AS c'
    ).single()['c']
    print('Pending TikTok summary embeddings:', pending)

mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=30_000)
mongo_coll = mongo_client[MONGO_DB][MONGO_COLLECTION]
mongo_client.admin.command('ping')
print('Mongo ok:', MONGO_DB, MONGO_COLLECTION)

print(f'Embedding provider: {EMBED_PROVIDER}  model/deployment: {EMBED_MODEL_NAME}  dim={EMBED_DIM}')


Neo4j ok: 1
Pending TikTok summary embeddings: 0


/tmp/ipykernel_96029/2650715156.py:49: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=30_000)


Mongo ok: rbl tiktok_user_video
Embedding provider: azure  model/deployment: text-embedding-3-large  dim=3072


## Video comment summary embedding functions

Defines summary fetch/write Cypher and the `embed_tiktok_comment_summaries` runner.

In [ ]:

def _write_summary_vector_chunk(driver, rows_chunk):
    """Small Neo4j transactions + retries (Aura / slow networks)."""
    from neo4j.exceptions import Neo4jError, ServiceUnavailable

    attempt = 0
    while True:
        try:
            with driver.session(database=NEO4J_DATABASE) as s:
                rec = s.run(WRITE_SUMMARY_CYPHER, rows=rows_chunk).single()
                return int(rec['written']) if rec else 0
        except (Neo4jError, ServiceUnavailable, TimeoutError, OSError) as e:
            attempt += 1
            if attempt > 8:
                raise
            wait = min(2 ** attempt, 120)
            print(f'  neo4j summary-vector write retry {attempt} after {wait}s: {e}')
            time.sleep(wait)


def _write_content_vector_chunk(driver, rows_chunk):
    from neo4j.exceptions import Neo4jError, ServiceUnavailable

    attempt = 0
    while True:
        try:
            with driver.session(database=NEO4J_DATABASE) as s:
                rec = s.run(WRITE_CONTENT_CYPHER, rows=rows_chunk).single()
                return int(rec['written']) if rec else 0
        except (Neo4jError, ServiceUnavailable, TimeoutError, OSError) as e:
            attempt += 1
            if attempt > 8:
                raise
            wait = min(2 ** attempt, 120)
            print(f'  neo4j content-vector write retry {attempt} after {wait}s: {e}')
            time.sleep(wait)

def ensure_vector_indexes(driver, dim: int) -> None:
    summary_stmt = f'''CREATE VECTOR INDEX tiktok_video_summary_embedding_index IF NOT EXISTS
    FOR (v:TikTokVideo) ON (v.comment_summary_embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}'''
    content_stmt = f'''CREATE VECTOR INDEX tiktok_video_content_embedding_index IF NOT EXISTS
    FOR (v:TikTokVideo) ON (v.video_content_embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}'''
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(summary_stmt)
        s.run(content_stmt)
    print('Ensured tiktok_video_summary_embedding_index and tiktok_video_content_embedding_index')


def embed_texts(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


FETCH_SUMMARY_CYPHER = '''
MATCH (v:TikTokVideo)
WHERE v.comment_summary_description IS NOT NULL
  AND v.comment_summary_embedding IS NULL
RETURN elementId(v) AS eid, v.comment_summary_description AS text
LIMIT $limit
'''

WRITE_SUMMARY_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE elementId(v) = row.eid
CALL db.create.setNodeVectorProperty(v, 'comment_summary_embedding', row.embedding)
RETURN count(*) AS written
'''

WRITE_CONTENT_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE elementId(v) = row.eid
SET v.video_content_text = row.text
WITH v, row
CALL db.create.setNodeVectorProperty(v, 'video_content_embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_tiktok_comment_summaries(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(FETCH_SUMMARY_CYPHER, limit=page_size))
        if not pending:
            break

        texts = [r['text'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [{'eid': pending[i]['eid'], 'embedding': embeddings[i]} for i in range(len(pending))]

        for chunk in batched(rows, SUMMARY_VECTOR_WRITE_BATCH):
            written = _write_summary_vector_chunk(driver, chunk)
            total_written += written

        print(f'  summary written={total_written}  elapsed={time.time()-start:.1f}s')

    return total_written


In [ ]:
# Summary status check (under comment summary section)
with driver.session(database=NEO4J_DATABASE) as s:
    summary_totals = {
        'videos_with_summary': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL RETURN count(v) AS c').single()['c'],
        'videos_with_summary_embedding': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c').single()['c'],
        'summary_pending': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL AND v.comment_summary_embedding IS NULL RETURN count(v) AS c').single()['c'],
    }

print(summary_totals)


{'videos_with_summary': 17325, 'videos_with_summary_embedding': 17325, 'summary_pending': 0}


## Run comment summary embeddings (embedding pass)

Run the **next code cell** to populate `comment_summary_embedding`.


In [ ]:
# === COMMENT SUMMARY EMBEDDING PASS ===
ensure_vector_indexes(driver, EMBED_DIM)
summary_written = embed_tiktok_comment_summaries(driver)
print('Done. Newly embedded TikTok comment summaries:', summary_written)

with driver.session(database=NEO4J_DATABASE) as s:
    summary_totals = {
        'videos_with_summary': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL RETURN count(v) AS c').single()['c'],
        'videos_with_summary_embedding': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c').single()['c'],
        'summary_pending': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL AND v.comment_summary_embedding IS NULL RETURN count(v) AS c').single()['c'],
    }
print(summary_totals)


Ensured tiktok_video_summary_embedding_index and tiktok_video_content_embedding_index
Done. Newly embedded TikTok comment summaries: 0
{'videos_with_summary': 17325, 'videos_with_summary_embedding': 17325, 'summary_pending': 0}


## Video content embedding functions

Defines helper text construction, content fetch/write Cypher, and the `embed_tiktok_video_content` runner.

In [ ]:
# TikTok video content embedding helpers and cypher

def format_sticker_info_list(val: Any) -> Optional[str]:
    if val is None:
        return None
    if isinstance(val, str):
        return val.strip() or None
    if isinstance(val, list) and len(val) == 0:
        return None
    try:
        return json.dumps(val, ensure_ascii=False)
    except (TypeError, ValueError):
        return str(val)


def build_tiktok_video_content_text(
    title: Optional[str],
    thumbnail_description: Optional[str],
    thumbnail_keywords: List[str],
    hashtag_names: List[str],
    description: Optional[str],
    voice_to_text: Optional[str],
    sticker_info_list: Any,
) -> Optional[str]:
    parts: List[str] = []
    if title:
        parts.append(f'Title: {title}')
    if thumbnail_description:
        parts.append(f'Thumbnail: {thumbnail_description}')
    if thumbnail_keywords:
        parts.append(f'Thumbnail keywords: {", ".join(thumbnail_keywords)}')
    if hashtag_names:
        parts.append(f'Hashtags: {", ".join(hashtag_names)}')
    if description:
        parts.append(f'Description: {description}')
    if voice_to_text:
        parts.append(f'Voice to text: {voice_to_text}')
    sticker_str = format_sticker_info_list(sticker_info_list)
    if sticker_str:
        parts.append(f'Stickers: {sticker_str}')
    return '\n'.join(parts).strip() if parts else None


FETCH_CONTENT_CYPHER = '''
MATCH (v:TikTokVideo)
WHERE v.video_content_embedding IS NULL
  AND (
    v.video_title IS NOT NULL
    OR v.video_thumbnail_description IS NOT NULL
    OR size(coalesce(v.video_thumbnail_keywords, [])) > 0
    OR EXISTS { (v)-[:HAS_TAG]->(:TikTokHashTag) }
    OR v.video_description IS NOT NULL
    OR v.voice_to_text IS NOT NULL
    OR EXISTS { (v)-[:HAS_STICKER]->(:TikTokSticker) }
  )
OPTIONAL MATCH (v)-[:HAS_TAG]->(ht:TikTokHashTag)
WITH v,
     elementId(v) AS eid,
     v.video_title AS video_title,
     v.video_thumbnail_description AS video_thumbnail_description,
     coalesce(v.video_thumbnail_keywords, []) AS video_thumbnail_keywords,
     v.video_description AS video_description,
     v.voice_to_text AS voice_to_text,
     [x IN collect(DISTINCT ht.name) WHERE x IS NOT NULL] AS hashtag_names_from_rels
OPTIONAL MATCH (v)-[:HAS_STICKER]->(st:TikTokSticker)
WITH eid,
     video_title,
     video_thumbnail_description,
     video_thumbnail_keywords,
     video_description,
     voice_to_text,
     hashtag_names_from_rels,
     [x IN collect(DISTINCT st.name) WHERE x IS NOT NULL] AS sticker_names_from_rels
RETURN eid,
       video_title,
       video_thumbnail_description,
       video_thumbnail_keywords,
       hashtag_names_from_rels,
       video_description,
       voice_to_text,
       sticker_names_from_rels
LIMIT $limit
'''


def embed_tiktok_video_content(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(FETCH_CONTENT_CYPHER, limit=page_size))
        if not pending:
            break

        rows_to_embed = []
        texts = []
        for record in pending:
            hn = record['hashtag_names_from_rels'] or []
            hashtag_names = [h for h in hn if isinstance(h, str) and h]
            sn = record['sticker_names_from_rels'] or []
            sticker_payload = [x for x in sn if isinstance(x, str) and x] or None
            tk = [k for k in (record['video_thumbnail_keywords'] or []) if isinstance(k, str) and k]
            text = build_tiktok_video_content_text(
                record['video_title'],
                record['video_thumbnail_description'],
                tk,
                hashtag_names,
                record['video_description'],
                record['voice_to_text'],
                sticker_payload,
            )
            if text:
                rows_to_embed.append({'eid': record['eid'], 'text': text})
                texts.append(text)

        if not rows_to_embed:
            break

        embeddings = embed_texts(texts)
        write_rows = [
            {'eid': rows_to_embed[i]['eid'], 'text': rows_to_embed[i]['text'], 'embedding': embeddings[i]}
            for i in range(len(rows_to_embed))
        ]

        for chunk in batched(write_rows, CONTENT_VECTOR_WRITE_BATCH):
            total_written += _write_content_vector_chunk(driver, chunk)

        print(f'  content written={total_written}  elapsed={time.time()-start:.1f}s')

    return total_written


## Run video content embedding

Runs only `video_content_embedding` creation and reports content embedding totals.

In [ ]:
# Run video content embeddings only (comment summary already completed)
with driver.session(database=NEO4J_DATABASE) as s:
    s.run(
        f'''CREATE VECTOR INDEX tiktok_video_content_embedding_index IF NOT EXISTS
        FOR (v:TikTokVideo) ON (v.video_content_embedding)
        OPTIONS {{ indexConfig: {{ `vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine' }} }}'''
    )

content_written = embed_tiktok_video_content(driver)
print('Done. Newly embedded TikTok video content:', content_written)

with driver.session(database=NEO4J_DATABASE) as s:
    content_totals = {
        'videos_with_content_embedding': s.run('MATCH (v:TikTokVideo) WHERE v.video_content_embedding IS NOT NULL RETURN count(v) AS c').single()['c'],
        'content_pending': s.run("MATCH (v:TikTokVideo) WHERE v.video_content_embedding IS NULL AND (v.video_title IS NOT NULL OR v.video_thumbnail_description IS NOT NULL OR size(coalesce(v.video_thumbnail_keywords, [])) > 0 OR EXISTS { (v)-[:HAS_TAG]->(:TikTokHashTag) } OR v.video_description IS NOT NULL OR v.voice_to_text IS NOT NULL OR EXISTS { (v)-[:HAS_STICKER]->(:TikTokSticker) }) RETURN count(v) AS c").single()['c'],
    }

print(content_totals)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `sticker_info_list` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=10, offset=280>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 280, 'line': 10, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_embedding IS NULL\n  AND (\n    v.video_title IS NOT NULL\n    OR v.video_thumbnail_description IS NOT NULL\n    OR size(coalesce(v.hashtag_names, [])) > 0\n    OR v.video_description IS NOT NULL\n    OR v.voice_to_text IS NOT NULL\n    OR v.sticker_info_list IS NOT NULL\n 

  content written=498  elapsed=42.8s


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `sticker_info_list` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=10, offset=280>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 280, 'line': 10, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_embedding IS NULL\n  AND (\n    v.video_title IS NOT NULL\n    OR v.video_thumbnail_description IS NOT NULL\n    OR size(coalesce(v.hashtag_names, [])) > 0\n    OR v.video_description IS NOT NULL\n    OR v.voice_to_text IS NOT NULL\n    OR v.sticker_info_list IS NOT NULL\n 

  content written=996  elapsed=84.2s


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `sticker_info_list` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=10, offset=280>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 280, 'line': 10, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_embedding IS NULL\n  AND (\n    v.video_title IS NOT NULL\n    OR v.video_thumbnail_description IS NOT NULL\n    OR size(coalesce(v.hashtag_names, [])) > 0\n    OR v.video_description IS NOT NULL\n    OR v.voice_to_text IS NOT NULL\n    OR v.sticker_info_list IS NOT NULL\n 

  content written=1491  elapsed=124.7s


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `sticker_info_list` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=10, offset=280>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 280, 'line': 10, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_embedding IS NULL\n  AND (\n    v.video_title IS NOT NULL\n    OR v.video_thumbnail_description IS NOT NULL\n    OR size(coalesce(v.hashtag_names, [])) > 0\n    OR v.video_description IS NOT NULL\n    OR v.voice_to_text IS NOT NULL\n    OR v.sticker_info_list IS NOT NULL\n 

## Comment topics — Mongo -> Neo4j (`TikTokCommentTopic`)

Reads `comments_frequent_topics` (+ categories / weights) from `tiktok_user_video`.
For each document, only if `(:TikTokVideo {video_id})` exists in Neo4j with the same numeric `video_id` as Mongo, existing `HAS_COMMENT_TOPIC` relationships from that video are removed and recreated pointing at `(:TikTokCommentTopic {video_id, name})`.

Videos that exist in Mongo but not yet in Neo4j are skipped (no orphan topic relationships).


In [ ]:
TIKTOK_TOPICS_MONGO_PROJECTION = {
    'video_id': 1,
    'comments_frequent_topics': 1,
    'comments_frequent_topic_categories': 1,
    'comments_frequent_topic_weights': 1,
}


def clean_text(value: Any) -> Optional[str]:
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return str(value)


def topic_row_from_mongo_doc(doc: dict) -> Optional[Dict[str, Any]]:
    """One Mongo doc → {video_id: int, topics: [...]} — same fields as tiktok_embedding_test.normalize_mongo_doc topics slice."""
    raw_id = doc.get('video_id')
    if raw_id is None:
        return None
    if isinstance(raw_id, dict) and '$numberLong' in raw_id:
        raw_id = raw_id['$numberLong']
    try:
        video_id = int(str(raw_id).strip())
    except (TypeError, ValueError):
        return None
    names = doc.get('comments_frequent_topics') or []
    categories = doc.get('comments_frequent_topic_categories') or []
    weights = doc.get('comments_frequent_topic_weights') or []
    topics: List[Dict[str, Any]] = []
    for i, name in enumerate(names):
        name = clean_text(name)
        if not name:
            continue
        topics.append(
            {
                'name': name,
                'category': clean_text(categories[i] if i < len(categories) else None),
                'weight': float(weights[i]) if i < len(weights) and weights[i] is not None else None,
                'position': i,
            }
        )
    return {'video_id': video_id, 'topics': topics}


SYNC_TIKTOK_COMMENT_TOPICS_CYPHER = """
UNWIND $rows AS row
OPTIONAL MATCH (v:TikTokVideo {video_id: row.video_id})
WITH row, v
WHERE v IS NOT NULL
OPTIONAL MATCH (v)-[r:HAS_COMMENT_TOPIC]->()
DELETE r
WITH row, v
UNWIND row.topics AS topic
WITH row, v, topic
WHERE topic.name IS NOT NULL
MERGE (ct:TikTokCommentTopic {video_id: row.video_id, name: topic.name})
SET ct.category = topic.category,
    ct.platform = $tiktok_platform
MERGE (v)-[rel:HAS_COMMENT_TOPIC]->(ct)
SET rel.weight = topic.weight,
    rel.position = topic.position,
    rel.platform = $tiktok_platform
"""


def ensure_tiktok_topic_constraint(driver) -> None:
    stmt = (
        'CREATE CONSTRAINT tiktok_comment_topic_video_name IF NOT EXISTS '
        'FOR (t:TikTokCommentTopic) REQUIRE (t.video_id, t.name) IS UNIQUE'
    )
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(stmt)
    print('Ensured constraint on (TikTokCommentTopic.video_id, TikTokCommentTopic.name)')


def ensure_tiktok_topic_vector_index(driver, dim: int) -> None:
    stmt = f"""CREATE VECTOR INDEX tiktok_comment_topic_embedding_index IF NOT EXISTS
    FOR (t:TikTokCommentTopic) ON (t.embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}"""
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(stmt)
    print('Ensured tiktok_comment_topic_embedding_index')


def sync_tiktok_comment_topics_from_mongo(
    driver,
    collection,
    batch_size: int = TOPIC_UPSERT_BATCH,
) -> tuple[int, int]:
    """Returns (mongo_docs_in_batches, topic_slot_rows_synced) — count of topic entries processed (≤10×videos when capped)."""
    cur = collection.find({}, TIKTOK_TOPICS_MONGO_PROJECTION)
    if TOPIC_SYNC_LIMIT > 0:
        cur = cur.limit(TOPIC_SYNC_LIMIT)
    total_docs = 0
    total_links = 0
    for batch in batched(cur, batch_size):
        rows = []
        for doc in batch:
            row = topic_row_from_mongo_doc(doc)
            if row:
                rows.append(row)
        if not rows:
            continue
        total_docs += len(rows)
        batch_topic_slots = sum(len(r['topics']) for r in rows)
        with driver.session(database=NEO4J_DATABASE) as s:
            s.run(
                SYNC_TIKTOK_COMMENT_TOPICS_CYPHER,
                rows=rows,
                tiktok_platform=TIKTOK_PLATFORM,
            )
        total_links += batch_topic_slots
        print(f'  topic sync: mongo_rows={total_docs}  topic_slots_synced={total_links}')
    return total_docs, total_links


TIKTOK_TOPIC_FETCH_CYPHER = """
MATCH (t:TikTokCommentTopic)
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND t.embedding IS NULL AND t.name IS NOT NULL
RETURN elementId(t) AS eid, t.name AS name
LIMIT $limit
"""

TIKTOK_TOPIC_WRITE_CYPHER = """
UNWIND $rows AS row
MATCH (t) WHERE elementId(t) = row.eid
CALL db.create.setNodeVectorProperty(t, 'embedding', row.embedding)
RETURN count(*) AS written
"""


def _embed_texts_for_tiktok_topics(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    """Same logic as `embed_texts` in the summary section — uses embed_client from setup so topic embedding works without running summary cells."""
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


def _write_tiktok_topic_embedding_chunk(driver, rows_chunk):
    """Write topic vectors in small Neo4j transactions; retry on transient/store errors (Aura)."""
    from neo4j.exceptions import Neo4jError

    attempt = 0
    while True:
        try:
            with driver.session(database=NEO4J_DATABASE) as s:
                rec = s.run(TIKTOK_TOPIC_WRITE_CYPHER, rows=rows_chunk).single()
                return int(rec['written']) if rec else 0
        except Neo4jError as e:
            attempt += 1
            if attempt > 8:
                raise
            wait = min(2 ** attempt, 120)
            print(f'  neo4j vector write retry {attempt} after {wait}s: {e}')
            time.sleep(wait)

def embed_tiktok_topic_nodes(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(
                s.run(
                    TIKTOK_TOPIC_FETCH_CYPHER,
                    limit=page_size,
                    platform=TIKTOK_PLATFORM,
                )
            )
        if not pending:
            break
        texts = [r['name'] for r in pending]
        embeddings = _embed_texts_for_tiktok_topics(texts)
        rows = [
            {'eid': pending[i]['eid'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, TOPIC_EMBED_WRITE_BATCH):
            written = _write_tiktok_topic_embedding_chunk(driver, chunk)
            total_written += written
        print(f'  TikTokCommentTopic embeddings written={total_written}  elapsed={time.time() - start:.1f}s')
    return total_written



In [ ]:
# Sync topics from Mongo, then embed TikTokCommentTopic.name
# Requires: setup cell (`driver`, `mongo_coll`, `embed_client`, `EMBED_DIM`) and — if you want summary/content vectors — those sections above.

ensure_tiktok_topic_constraint(driver)
mongo_docs, topic_links = sync_tiktok_comment_topics_from_mongo(driver, mongo_coll)
print('Topic sync done. mongo_docs (batched rows):', mongo_docs, '  topic_slots:', topic_links)

ensure_tiktok_topic_vector_index(driver, EMBED_DIM)
topic_embed_written = embed_tiktok_topic_nodes(driver)
print('Topic embedding rows written:', topic_embed_written)

with driver.session(database=NEO4J_DATABASE) as s:
    stats = {
        'TikTokCommentTopic': s.run('MATCH (t:TikTokCommentTopic) RETURN count(t) AS c').single()['c'],
        'HAS_COMMENT_TOPIC': s.run('MATCH ()-[r:HAS_COMMENT_TOPIC]->(:TikTokCommentTopic) RETURN count(r) AS c').single()['c'],
        'topics_with_embedding': s.run(
            "MATCH (t:TikTokCommentTopic) WHERE t.embedding IS NOT NULL RETURN count(t) AS c"
        ).single()['c'],
    }
print(stats)


In [ ]:
_d = globals().get('driver')
if _d is not None:
    try:
        _d.close()
    except Exception:
        pass
    print('Closed Neo4j driver')
else:
    print('No Neo4j driver in memory - run the setup cell that creates `driver` first.')

_mc = globals().get('mongo_client')
if _mc is not None:
    try:
        _mc.close()
    except Exception:
        pass
    print('Closed Mongo client')
else:
    print('No Mongo client in memory - skipped.')
